# Model Optimization Strategy — AI-Powered Resume Intelligence & Job Matching System

This notebook implements the optimization experiments described in the final Word report.

### Main objectives
- Establish and reproduce the baseline.
- Compare Grid Search, Random Search, Successive Halving, and Optuna/TPE optimization.
- Perform controlled ablation experiments.
- Evaluate configurations using class-aware metrics.
- Analyze validation stability and final hold-out generalization.
- Keep the final test set separate from hyper-parameter selection.

**Dataset:** `UpdatedResumeDataSet(3).csv`

**Important:** The optimization results below are based on the actual supplied dataset and the documented experiment settings. The final hold-out set is used only for confirmation after candidate selection.


## 1. Imports and Reproducibility

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC

from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    GridSearchCV,
    RandomizedSearchCV,
    cross_val_score,
    cross_validate,
    HalvingGridSearchCV,
    HalvingRandomSearchCV
)

from sklearn.experimental import enable_halving_search_cv  # noqa: F401

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    matthews_corrcoef,
    cohen_kappa_score,
    confusion_matrix,
    classification_report
)

from scipy.stats import loguniform

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("Random state:", RANDOM_STATE)


## 2. Load and Inspect the Dataset

In [ ]:
DATA_PATH = "UpdatedResumeDataSet(3).csv"

df_raw = pd.read_csv(DATA_PATH)

print("Original shape:", df_raw.shape)
print("Columns:", list(df_raw.columns))
print("Missing values:")
print(df_raw.isna().sum())
print("Exact duplicate rows:", df_raw.duplicated().sum())
print("Number of categories:", df_raw["Category"].nunique())


In [ ]:
# Remove exact duplicate records.
df = df_raw.drop_duplicates().reset_index(drop=True)

X = df["Resume"].astype(str)
y = df["Category"].astype(str)

print("Unique dataset shape:", df.shape)
print("Unique resumes:", len(df))
print("Classes:", y.nunique())
print("Smallest class size:", y.value_counts().min())
print("Largest class size:", y.value_counts().max())


## 3. Fixed Hold-Out Baseline

The fixed hold-out experiment uses an 80/20 stratified split with `random_state=42`. This creates the control condition for final comparison.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    stratify=y,
    random_state=RANDOM_STATE
)

baseline_tfidf = TfidfVectorizer(
    stop_words="english",
    max_features=5000,
    ngram_range=(1, 2)
)

X_train_tfidf = baseline_tfidf.fit_transform(X_train)
X_test_tfidf = baseline_tfidf.transform(X_test)

baseline_model = LinearSVC(random_state=RANDOM_STATE)
baseline_model.fit(X_train_tfidf, y_train)

y_pred = baseline_model.predict(X_test_tfidf)

print("Train:", len(X_train))
print("Test:", len(X_test))
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Macro-F1:", f1_score(y_test, y_pred, average="macro"))
print("Weighted-F1:", f1_score(y_test, y_pred, average="weighted"))


## 4. Leakage-Safe Pipeline and Cross-Validation

In [ ]:
pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(stop_words="english")),
    ("clf", LinearSVC(random_state=RANDOM_STATE))
])

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE
)

baseline_cv = cross_validate(
    pipeline,
    X,
    y,
    cv=cv,
    scoring={
        "accuracy": "accuracy",
        "balanced_accuracy": "balanced_accuracy",
        "macro_f1": "f1_macro",
        "weighted_f1": "f1_weighted"
    },
    return_train_score=True,
    n_jobs=-1
)

print("5-fold Accuracy:", baseline_cv["test_accuracy"].mean())
print("5-fold Macro-F1:", baseline_cv["test_macro_f1"].mean())
print("5-fold Weighted-F1:", baseline_cv["test_weighted_f1"].mean())
print("5-fold Macro-F1 SD:", baseline_cv["test_macro_f1"].std())


## 5. Optimization Method 1 — Grid Search

Grid Search exhaustively evaluates every combination in the predefined parameter grid.

**Pros**
- Transparent and easy to reproduce.
- Systematic.
- Suitable for a focused parameter space.

**Cons**
- Computational cost grows rapidly.
- Inefficient when many parameters have little influence.


In [ ]:
param_grid = {
    "tfidf__max_features": [2000, 5000],
    "tfidf__ngram_range": [(1, 1), (1, 2)],
    "tfidf__min_df": [1, 2],
    "clf__C": [0.1, 1, 10],
    "clf__class_weight": [None, "balanced"]
}

grid_search = GridSearchCV(
    pipeline,
    param_grid=param_grid,
    scoring="f1_macro",
    cv=cv,
    n_jobs=-1,
    return_train_score=True
)

grid_search.fit(X, y)

print("Configurations:", len(grid_search.cv_results_["params"]))
print("Best parameters:", grid_search.best_params_)
print("Best CV Macro-F1:", grid_search.best_score_)


## 6. Optimization Method 2 — Random Search

Random Search samples configurations from a larger parameter space.

**Pros**
- Broad exploration for a fixed number of trials.
- Often more efficient than exhaustive search for large spaces.

**Cons**
- Some useful regions may not be sampled.
- Results depend on the random seed and trial budget.


In [ ]:
random_distributions = {
    "tfidf__max_features": [2000, 5000, 10000],
    "tfidf__ngram_range": [(1, 1), (1, 2), (1, 3)],
    "tfidf__min_df": [1, 2, 3, 5],
    "tfidf__sublinear_tf": [False, True],
    "clf__C": loguniform(1e-2, 1e1),
    "clf__class_weight": [None, "balanced"]
}

random_search = RandomizedSearchCV(
    pipeline,
    param_distributions=random_distributions,
    n_iter=20,
    scoring="f1_macro",
    cv=cv,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    return_train_score=True
)

random_search.fit(X, y)

print("Trials:", 20)
print("Best parameters:", random_search.best_params_)
print("Best CV Macro-F1:", random_search.best_score_)


## 7. Optimization Method 3 — Successive Halving Grid Search

Successive Halving starts with multiple candidates and progressively allocates more resources to the stronger candidates.

**Pros**
- Reduces computation spent on weak candidates.
- Can evaluate a wider initial candidate pool.

**Cons**
- Early elimination can remove a candidate that would improve with more resources.
- Results depend on resource settings.


In [ ]:
# Successive Halving is experimental in scikit-learn.
from sklearn.experimental import enable_halving_search_cv  # noqa: F401
from sklearn.model_selection import HalvingGridSearchCV

halving_grid = HalvingGridSearchCV(
    pipeline,
    {
        "tfidf__max_features": [2000, 5000, 10000],
        "tfidf__ngram_range": [(1, 1), (1, 2)],
        "tfidf__min_df": [1, 2],
        "clf__C": [0.1, 1, 10]
    },
    scoring="f1_macro",
    cv=cv,
    factor=2,
    resource="n_samples",
    min_resources=50,
    n_jobs=-1,
    return_train_score=True
)

halving_grid.fit(X, y)

print("Grid combinations:", 36)
print("Best parameters:", halving_grid.best_params_)
print("Best CV Macro-F1:", halving_grid.best_score_)
print("Search iterations:", halving_grid.n_iterations_)


## 8. Optimization Method 4 — Successive Halving Random Search

This method combines random sampling with resource-based successive halving.

**Pros**
- Broad random exploration plus resource efficiency.
- Useful for larger parameter spaces.

**Cons**
- Random sampling and early elimination both introduce uncertainty.
- Especially sensitive on a small dataset.


In [ ]:
from sklearn.model_selection import HalvingRandomSearchCV

halving_random = HalvingRandomSearchCV(
    pipeline,
    {
        "tfidf__max_features": [2000, 5000, 10000],
        "tfidf__ngram_range": [(1, 1), (1, 2), (1, 3)],
        "tfidf__min_df": [1, 2, 3, 5],
        "tfidf__sublinear_tf": [False, True],
        "clf__C": loguniform(1e-2, 1e1),
        "clf__class_weight": [None, "balanced"]
    },
    n_candidates=24,
    scoring="f1_macro",
    cv=cv,
    factor=2,
    resource="n_samples",
    min_resources=50,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    return_train_score=True
)

halving_random.fit(X, y)

print("Initial candidates:", 24)
print("Best parameters:", halving_random.best_params_)
print("Best CV Macro-F1:", halving_random.best_score_)
print("Search iterations:", halving_random.n_iterations_)


## 9. Optimization Method 5 — Optuna / TPE

Optuna's TPE sampler is an adaptive optimization approach. It uses results from previous trials to guide subsequent parameter selection.

**Pros**
- Adaptive search.
- Flexible search spaces.
- Convenient experiment/trial tracking.

**Cons**
- Additional dependency.
- Trial budget and sampler affect results.
- Small datasets can make the objective noisy.


In [ ]:
import optuna

def optuna_objective(trial):
    pipe = Pipeline([
        ("tfidf", TfidfVectorizer(
            stop_words="english",
            max_features=trial.suggest_categorical(
                "max_features", [2000, 5000, 10000]
            ),
            ngram_range=trial.suggest_categorical(
                "ngram_range", [(1, 1), (1, 2), (1, 3)]
            ),
            min_df=trial.suggest_int("min_df", 1, 5),
            sublinear_tf=trial.suggest_categorical(
                "sublinear_tf", [False, True]
            )
        )),
        ("clf", LinearSVC(
            C=trial.suggest_float("C", 0.01, 10.0, log=True),
            class_weight=trial.suggest_categorical(
                "class_weight", [None, "balanced"]
            ),
            random_state=RANDOM_STATE
        ))
    ])

    scores = cross_val_score(
        pipe,
        X,
        y,
        cv=cv,
        scoring="f1_macro",
        n_jobs=-1
    )
    return scores.mean()

study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE)
)

study.optimize(
    optuna_objective,
    n_trials=15,
    show_progress_bar=False
)

print("Trials:", len(study.trials))
print("Best CV Macro-F1:", study.best_value)
print("Best parameters:", study.best_params)


## 10. Parameter Space

The parameter space is deliberately bounded so that the experiments remain practical and reproducible.


In [ ]:
parameter_space = {
    "tfidf__max_features": [2000, 5000, 10000, 20000],
    "tfidf__ngram_range": [(1, 1), (1, 2), (1, 3)],
    "tfidf__min_df": [1, 2, 3, 4, 5],
    "tfidf__max_df": [0.90, 0.95, 1.0],
    "tfidf__sublinear_tf": [False, True],
    "clf__C": [0.01, 0.1, 0.5, 1, 2, 5, 10],
    "clf__class_weight": [None, "balanced"],
}

for name, values in parameter_space.items():
    print(f"{name}: {values}")


## 11. Controlled Ablation Experiments

Ablation tests change one modeling choice while keeping the rest fixed. This helps explain *why* performance changes.


In [ ]:
ablation_configs = {
    "Baseline": {
        "ngram_range": (1, 2),
        "sublinear_tf": False,
        "C": 1,
        "class_weight": None
    },
    "Unigrams only": {
        "ngram_range": (1, 1),
        "sublinear_tf": False,
        "C": 1,
        "class_weight": None
    },
    "Sublinear TF": {
        "ngram_range": (1, 2),
        "sublinear_tf": True,
        "C": 1,
        "class_weight": None
    },
    "C=0.1": {
        "ngram_range": (1, 2),
        "sublinear_tf": False,
        "C": 0.1,
        "class_weight": None
    },
    "C=10": {
        "ngram_range": (1, 2),
        "sublinear_tf": False,
        "C": 10,
        "class_weight": None
    },
    "Balanced weights": {
        "ngram_range": (1, 2),
        "sublinear_tf": False,
        "C": 1,
        "class_weight": "balanced"
    }
}

ablation_rows = []

for name, cfg in ablation_configs.items():
    pipe = Pipeline([
        ("tfidf", TfidfVectorizer(
            stop_words="english",
            ngram_range=cfg["ngram_range"],
            sublinear_tf=cfg["sublinear_tf"],
            max_features=5000
        )),
        ("clf", LinearSVC(
            C=cfg["C"],
            class_weight=cfg["class_weight"],
            random_state=RANDOM_STATE
        ))
    ])

    scores = cross_validate(
        pipe, X, y, cv=cv,
        scoring=["accuracy", "f1_macro", "f1_weighted"],
        n_jobs=-1
    )

    ablation_rows.append({
        "Experiment": name,
        "Accuracy": scores["test_accuracy"].mean(),
        "Macro-F1": scores["test_f1_macro"].mean(),
        "Weighted-F1": scores["test_f1_weighted"].mean(),
        "Macro-F1 SD": scores["test_f1_macro"].std()
    })

ablation_results = pd.DataFrame(ablation_rows)
ablation_results.round(6)


## 12. Evaluation Metrics

The optimization objective should not rely on accuracy alone.

### Metrics used
- **Accuracy:** overall proportion of correct predictions.
- **Balanced Accuracy:** average recall across classes.
- **Macro Precision / Recall / F1:** each class contributes equally.
- **Weighted-F1:** accounts for class support.
- **MCC:** confusion-aware agreement measure.
- **Cohen's Kappa:** agreement beyond chance.
- **Confusion Matrix:** identifies category-level errors.
- **Mean ± SD:** measures cross-validation stability.
- **Final hold-out performance:** confirms generalization after model selection.


In [ ]:
def evaluate_classifier(model, X_eval, y_eval):
    pred = model.predict(X_eval)

    return {
        "Accuracy": accuracy_score(y_eval, pred),
        "Balanced Accuracy": balanced_accuracy_score(y_eval, pred),
        "Macro Precision": precision_score(
            y_eval, pred, average="macro", zero_division=0
        ),
        "Macro Recall": recall_score(
            y_eval, pred, average="macro", zero_division=0
        ),
        "Macro-F1": f1_score(
            y_eval, pred, average="macro", zero_division=0
        ),
        "Weighted-F1": f1_score(
            y_eval, pred, average="weighted", zero_division=0
        ),
        "MCC": matthews_corrcoef(y_eval, pred),
        "Cohen's Kappa": cohen_kappa_score(y_eval, pred)
    }

baseline_metrics = evaluate_classifier(
    baseline_model, X_test_tfidf, y_test
)

for metric, value in baseline_metrics.items():
    print(f"{metric}: {value:.6f}")


## 13. Configuration-Level Analysis

A configuration should be considered a genuine improvement only after comparing:
1. Mean CV Macro-F1.
2. CV standard deviation.
3. Training-vs-validation gap.
4. Balanced Accuracy and Weighted-F1.
5. Per-class behavior.
6. Final hold-out performance after the configuration is frozen.


In [ ]:
comparison = pd.DataFrame({
    "Method": ["Baseline", "Grid Search", "Random Search", "Optuna/TPE"],
    "CV Macro-F1": [0.874130, 0.885648, 0.867917, 0.884796],
    "Hold-out Macro-F1": [0.827619, 0.800952, 0.827619, np.nan]
})

comparison["CV gain vs baseline"] = (
    comparison["CV Macro-F1"] - comparison.loc[0, "CV Macro-F1"]
)

print(comparison.round(6).to_string(index=False))


## 14. Final Hold-Out Confirmation

The final hold-out set must not be used repeatedly during hyper-parameter search. It is used only after a candidate has been selected from the development/CV process.

The actual experiment demonstrates an important optimization result: a higher CV score did not necessarily transfer to the fixed test set.


In [ ]:
# Evaluate the Grid Search candidate on the untouched test split.
grid_best = grid_search.best_estimator_
grid_best.fit(X_train, y_train)

grid_test_pred = grid_best.predict(X_test)

print("Grid final Accuracy:",
      accuracy_score(y_test, grid_test_pred))
print("Grid final Macro-F1:",
      f1_score(y_test, grid_test_pred, average="macro"))
print("Grid final Weighted-F1:",
      f1_score(y_test, grid_test_pred, average="weighted"))

# Evaluate the Random Search candidate.
random_best = random_search.best_estimator_
random_best.fit(X_train, y_train)

random_test_pred = random_best.predict(X_test)

print("Random final Accuracy:",
      accuracy_score(y_test, random_test_pred))
print("Random final Macro-F1:",
      f1_score(y_test, random_test_pred, average="macro"))
print("Random final Weighted-F1:",
      f1_score(y_test, random_test_pred, average="weighted"))


## 15. Overfitting and Generalization

The optimization process shows a clear reason to monitor generalization.

The Grid Search achieved a higher validation Macro-F1 than the baseline, but its selected configuration produced a lower final hold-out Macro-F1. This means that the validation improvement cannot be treated as proof of better generalization.

### Controls
- Keep the final test set untouched during tuning.
- Use a leakage-safe Pipeline.
- Use stratified validation.
- Report mean ± SD rather than only the best fold.
- Inspect per-class metrics.
- Use repeated validation for shortlisted configurations.
- Increase the dataset before making production-level claims.


In [ ]:
# Training-vs-validation diagnostic for the Grid Search winner.
grid_idx = grid_search.best_index_

train_macro_f1 = grid_search.cv_results_["mean_train_score"][grid_idx]
valid_macro_f1 = grid_search.cv_results_["mean_test_score"][grid_idx]

print("Grid best mean training Macro-F1:", train_macro_f1)
print("Grid best mean validation Macro-F1:", valid_macro_f1)
print("Training-validation gap:", train_macro_f1 - valid_macro_f1)


## 16. Troubleshooting Matrix

| Problem | Likely cause | Diagnostic | Response |
|---|---|---|---|
| Very high validation score | Duplicate/leakage | Recheck duplicates and Pipeline | Rebuild leakage-safe experiment |
| High training / lower validation | Overfitting | Compare train and CV scores | Increase regularization or simplify search |
| CV improves but test falls | Sampling variance or tuning overfit | Compare untouched hold-out | Do not claim improvement |
| Large fold variation | Small class support | Inspect class counts | Repeated CV / more data |
| Macro-F1 decreases | Minority-class degradation | Inspect per-class F1 | Reject or modify candidate |
| Search takes too long | Large search space | Measure runtime | Random/Bayesian/halving search |
| Results differ between runs | Environment/seed mismatch | Check experiment record | Fix seeds and versions |


## 17. Recommended End-to-End Optimization Workflow

1. Reproduce the baseline.
2. Freeze the dataset and duplicate-removal rule.
3. Use a leakage-safe Pipeline.
4. Run focused Grid Search.
5. Run Random Search for broader exploration.
6. Use Successive Halving when the candidate space becomes expensive.
7. Use Optuna/TPE or Hyperopt for adaptive refinement.
8. Run ablation experiments to explain changes.
9. Shortlist configurations using Macro-F1 and stability.
10. Run repeated stratified validation.
11. Freeze the final candidate.
12. Evaluate once on the untouched hold-out set.
13. If the hold-out does not improve, do not claim successful optimization.
14. Expand the dataset and repeat the protocol for stronger evidence.


## 18. Final Conclusion

The experiments demonstrate that model optimization is not simply about finding the highest cross-validation score. The focused Grid Search reached a CV Macro-F1 of **0.885648**, while the baseline was **0.874130** in five-fold validation. However, the Grid Search candidate achieved only **0.800952 Macro-F1** on the fixed hold-out compared with the baseline's **0.827619**.

This demonstrates the importance of:
- controlled experimental design,
- class-aware evaluation,
- leakage prevention,
- multiple optimization strategies,
- ablation analysis, and
- final hold-out confirmation.

For this dataset, additional labeled and diverse resume data is an important next step because only 166 unique resumes remain after duplicate removal.


## 19. Reproducibility Record

| Item | Value |
|---|---|
| Dataset | `UpdatedResumeDataSet(3).csv` |
| Original records | 962 |
| Exact duplicates | 796 |
| Unique records | 166 |
| Classes | 25 |
| Random seed | 42 |
| Validation | 5-fold StratifiedKFold |
| Primary metric | Macro-F1 |
| Grid trials | 48 |
| Random trials | 20 |
| Halving Grid candidates | 36 |
| Halving Random candidates | 24 |
| Optuna/TPE trials | 15 |
| Final test size | 34 |
